<div align="center">

# CLIPPR

### Design a PPR protein that binds any RNA sequence you choose

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/CLIPPR_designer.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-1a7f5a.svg)](https://github.com/SyedZainAliShah/clippr/blob/main/LICENSE)
[![Tests](https://img.shields.io/badge/tests-283%20passing-1a7f5a.svg)](https://github.com/SyedZainAliShah/clippr)

**iGEM Marburg 2026**

</div>

<div align="center">

<img src="https://raw.githubusercontent.com/SyedZainAliShah/clippr/main/notebooks/pipeline.svg" alt="CLIPPR pipeline: a target RNA passes through ppr, arelf, overhangs, codons, assembly and qc to become orderable DNA fragments" width="100%" style="max-width:980px">

</div>

---

Pentatricopeptide repeat proteins are built from tandem ~31-residue repeats, and **each
repeat reads exactly one RNA base** through two specificity residues:

| 5th + last residue | reads |     | 5th + last residue | reads |
|:---:|:---:|---|:---:|:---:|
| `T` `N` | **A** |  | `T` `D` | **G** |
| `N` `N` | **C** |  | `N` `D` | **U** |

So the protein is a deterministic function of your target — no catalogue to search. Give it
nine bases and you get a nine-repeat protein, a synthesisable coding sequence, a Golden Gate
assembly plan, and the fragments to order.

**Run the cells top to bottom.** Only the *Design parameters* cell normally needs editing.

> ##### Before you read any number
> **Predicted fidelity** comes from published ligation-count matrices (Pryor *et al.* 2020) —
> it is not a measured assembly efficiency in your hands. **QC** is a sequence-complexity
> check, not calibrated against vendor outcomes. **Cost** is a list price, not a quote.
> Nothing here has been validated at the bench.

In [ ]:
#@title Setup — install CLIPPR {display-mode: "form"}
#@markdown Installs the package if it is not already available. Safe to re-run — it never
#@markdown reinstalls over a working copy.
REPO = "SyedZainAliShah/clippr"

try:
    import clippr
    _msg = f"clippr {clippr.__version__} already available"
except ImportError:
    token = None
    try:
        from google.colab import userdata          # private-repo fallback
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = (f"git+https://{token}@github.com/{REPO}.git" if token
           else f"git+https://github.com/{REPO}.git")
    %pip install --quiet $url
    import clippr
    _msg = f"installed clippr {clippr.__version__}"

from IPython.display import HTML, display
display(HTML(
    f'<div style="border-left:3px solid #1a7f5a;padding:.5em .9em;'
    f'font-family:ui-monospace,monospace;font-size:13px;opacity:.85">{_msg}</div>'))

In [ ]:
#@title Design parameters — edit these {display-mode: "form"}
#@markdown ### Target
#@markdown The RNA sequence the PPR should bind. **Its length sets the architecture** —
#@markdown 9, 14 or 19 bases give a 9S, 14S or 19S protein.
target_rna = "AAAAUGUGG"  #@param {type:"string"}

#@markdown ### Host
#@markdown The genetic code follows automatically — nuclear hosts use table 1, chloroplasts
#@markdown table 11. **The two Chlamydomonas entries are not interchangeable:** the nucleus
#@markdown is GC-rich and prefers Leu `CTG` (0.73); the chloroplast is AT-rich and prefers
#@markdown Leu `TTA` (0.74). Using one for the other produces DNA that looks fine and is wrong.
organism = "c_reinhardtii_nuclear"  #@param ["c_reinhardtii_nuclear", "c_reinhardtii_chloroplast", "e_coli", "s_cerevisiae", "a_thaliana_nuclear", "n_tabacum_chloroplast"]

#@markdown ### Or bring your own codon usage
#@markdown Three ways, in order of precedence. Leave all blank to use the host above.
#@markdown
#@markdown **A file** — a `codon,frequency` CSV or a CDS FASTA. Upload it with the folder
#@markdown icon in the sidebar, or run the upload cell below, then put the filename here.
codon_table_file = ""  #@param {type:"string"}
#@markdown **A Kazusa species ID** — any NCBI taxonomy id Kazusa carries, e.g. `4577` for
#@markdown maize. Fetched and cached on first use.
kazusa_taxid = 0  #@param {type:"integer"}
#@markdown **The genetic code** to go with a table you supplied. Leave 0 to inherit from
#@markdown the host. Set 11 for anything organellar — a nuclear code on a chloroplast
#@markdown construct produces DNA that looks fine and is wrong.
genetic_code_override = 0  #@param {type:"integer"}

#@markdown ### Which enzyme sites must be absent
#@markdown `assembly` — this assembly's own chemistry (BsaI, BbsI)
#@markdown &nbsp;&nbsp;·&nbsp; `igem_rfc1000` — adds SapI, required by iGEM's Type IIS standard
#@markdown &nbsp;&nbsp;·&nbsp; `moclo_compat` — adds BsmBI to keep later MoClo levels open,
#@markdown a preference that can make some junctions infeasible
enzyme_profile = "igem_rfc1000"  #@param ["assembly", "igem_rfc1000", "moclo_compat"]

#@markdown **Extra sites to keep clear** — anything else this particular experiment needs
#@markdown absent, beyond the profile. Comma-separated, any name Biopython knows.
#@markdown Examples: `EcoRI, BamHI, HindIII, NotI`.
extra_blacklist = ""  #@param {type:"string"}

#@markdown ### Assembly
#@markdown Which Type IIS enzyme cuts the fragments out, and which published mis-ligation
#@markdown table scores the junctions. `BsaI-HFv2` and `BbsI-HF` are the two measured in
#@markdown Pryor *et al.* 2020 at 25 °C over 18 h.
assembly_enzyme = "BsaI"  #@param ["BsaI", "BbsI", "BsmBI", "SapI"]
ligation_table = "BsaI-HFv2"  #@param ["BsaI-HFv2", "BbsI-HF"]

#@markdown ### Destination vector level
#@markdown Or type your own acceptor overhangs below as `5prime,3prime` coding sites —
#@markdown they override the level. Remember the 3' entry is the **coding site**; the
#@markdown enzyme leaves its reverse complement.
destination_level = "level0"  #@param ["level_minus1", "level0", "level1"]
custom_destination = ""  #@param {type:"string"}

#@markdown ### Fragments
#@markdown How many pieces to split the gene into. Leave at 0 to let the length decide —
#@markdown set it only if your vendor has an awkward limit.
n_fragments = 0  #@param {type:"integer"}

#@markdown ### Reproducibility
#@markdown The same seed always gives the same design.
seed = 42  #@param {type:"integer"}
write_files = True  #@param {type:"boolean"}
check_offtarget = True  #@param {type:"boolean"}

In [ ]:
#@title Upload a codon table (optional) {display-mode: "form"}
#@markdown Optional. Run this only if you want to upload a codon table from your computer
#@markdown rather than type a path. It puts the file in the runtime and fills in the
#@markdown filename for you — then re-run the Design cell.
try:
    from google.colab import files as _f
    _up = _f.upload()
    if _up:
        codon_table_file = list(_up)[0]
        print(f"using {codon_table_file}")
except ImportError:
    print("not running on Colab — put the file beside the notebook and give its path "
          "in codon_table_file instead.")

In [ ]:
#@title Design — run this {display-mode: "form"}
#@markdown Picks cut positions and Golden Gate overhangs **first**, then codon-optimises with
#@markdown those positions locked — optimising first would let the optimiser rewrite the very
#@markdown bases the junctions depend on.
from clippr import DESTINATION_OVERHANGS, design_oneshot, table_from_kazusa
from IPython.display import HTML, display

result = design_oneshot(
    target_rna,
    organism=organism,
    codon_table=(codon_table_file or
                 (table_from_kazusa(kazusa_taxid) if kazusa_taxid else None)),
    genetic_code=genetic_code_override or None,
    enzyme_profile=enzyme_profile,
    extra_blacklist=extra_blacklist,
    enzyme=assembly_enzyme,
    matrix=ligation_table,
    destination=(tuple(s.strip().upper() for s in custom_destination.split(",")[:2])
                 if custom_destination else DESTINATION_OVERHANGS[destination_level]),
    n_fragments=n_fragments or None,
    seed=seed,
    check_offtarget=check_offtarget,
    outdir="clippr_output" if write_files else None,
)

qc = result["qc"]
TONE = {"PASS": "#1a7f5a", "WARNING": "#9a6b1f", "FAIL": "#a8402c"}
tone = TONE.get(qc["status"], "#6b7b75")


def _stat(label, value, hint=""):
    return (
        f'<div style="padding:.55em .9em .6em;border-left:1px solid rgba(128,145,138,.35)">'
        f'<div style="font-size:10.5px;letter-spacing:.09em;text-transform:uppercase;'
        f'opacity:.6">{label}</div>'
        f'<div style="font-size:19px;font-weight:600;font-variant-numeric:tabular-nums;'
        f'margin-top:.15em">{value}</div>'
        f'<div style="font-size:11.5px;opacity:.6">{hint}</div></div>')


ppr_code = result["ppr_code"]
if len(ppr_code) > 24:
    ppr_code = ppr_code[:24] + "…"

cards = "".join([
    _stat("architecture", result["architecture"], f'{len(result["protein"])} aa protein'),
    _stat("coding sequence", f'{len(result["cds"])} nt', ppr_code),
    _stat("fragments", len(result["oligos"]),
          "cuts at " + ", ".join(str(c) for c in result["cuts"])),
    _stat("fidelity", f'{result["fidelity"]:.3f}', "predicted, not measured"),
    _stat("GC", f'{qc["gc_pct"]:.1f}%',
          f'windows {qc["gc_window_min"]:.0f}–{qc["gc_window_max"]:.0f}%'),
    _stat("repeats", f'{qc["repeated_kmer_fraction"]:.1%}',
          f'longest {qc["longest_repeat"]} nt'),
])

warn = "".join(
    f'<div style="border-left:3px solid {TONE["WARNING"]};padding:.5em .9em;'
    f'margin-top:.7em;font-size:13px;line-height:1.5">{w}</div>'
    for w in result["warnings"])

RULE = "1px solid rgba(128,145,138,.35)"
card = (
    f'<div style="font-family:ui-sans-serif,system-ui,sans-serif;border:{RULE};'
    f'border-radius:5px;overflow:hidden;max-width:920px">'
    f'<div style="display:flex;align-items:center;gap:.8em;padding:.7em 1em;'
    f'border-bottom:{RULE}">'
    f'<span style="font-family:ui-monospace,monospace;font-size:16px;font-weight:600">'
    f'{result["target_rna"]}</span>'
    f'<span style="background:{tone};color:#fff;font-size:11px;font-weight:700;'
    f'letter-spacing:.07em;padding:.2em .7em;border-radius:99px">{qc["status"]}</span>'
    f'<span style="margin-left:auto;font-size:12.5px;opacity:.65">'
    f'{result["cost"]["total_eur"]:.2f} EUR list price · not a quote</span></div>'
    f'<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr))">'
    f'{cards}</div></div>{warn}')

display(HTML(card))

## The fragments to order

One row per orderable piece. `oh5` and `oh3` are the four-base Golden Gate overhangs that
join each fragment to its neighbours.

In [ ]:
#@title Fragment table {display-mode: "form"}
cols = {"fragment_id": "fragment", "assembly_order": "order", "aa_length": "residues",
        "oligo_length": "oligo nt", "oh5_coding_site_5to3": "oh5",
        "oh3_coding_site_5to3": "oh3"}
table = result["oligos"][list(cols)].rename(columns=cols)

# pandas' .style needs jinja2, which Colab has but a bare local environment may not.
# Test that directly rather than catching the AttributeError pandas raises, which would also
# swallow real errors. Falls back to the plain frame: cosmetics should never break a cell.
try:
    import jinja2  # noqa: F401
    display(table.style.hide(axis="index").set_properties(
        subset=["oh5", "oh3"], **{"font-family": "ui-monospace, monospace"}).set_table_styles([
            {"selector": "th", "props": [("text-align", "left"), ("font-size", "11px"),
                                         ("letter-spacing", ".07em"),
                                         ("text-transform", "uppercase"),
                                         ("opacity", ".65"), ("padding", ".4em .9em")]},
            {"selector": "td", "props": [("padding", ".4em .9em"),
                                         ("font-variant-numeric", "tabular-nums")]}]))
except ImportError:
    display(table)

## Why this design, and not another

Every junction records the overhangs it *could* have used and what became of each:

- **selected** — the one used
- **considered** — feasible, but another scored at least as well
- **rejected** — no synonymous codon arrangement could avoid an excluded enzyme site, so
  that junction is *impossible* under the active profile, not merely worse

`local realizations` counts the synonymous arrangements still available around a junction.
It is reported, never used to choose — but a junction with 2 is more fragile than one with
16, and that is worth seeing before you order.

**If a design looks surprising, read this rather than trusting it.**

In [ ]:
#@title Design audit {display-mode: "form"}
audit = result["audit"]
print(audit.report())

rejected = audit.rejected
print(f"\n{len(rejected)} candidate overhang(s) ruled out entirely under "
      f"profile '{audit.enzyme_profile}'")
for d in rejected[:8]:
    print(f"    {d.sequence}  cut {d.junction_cut:>4d}   {d.reason}")
if len(rejected) > 8:
    print(f"    … and {len(rejected) - 8} more")
if not rejected:
    print("    (every achievable overhang was usable at every junction)")

## Take the files

Four artefacts: the order CSV, the oligos as FASTA, the assembled gene, and an annotated
GenBank record — every PPR repeat labelled with the base it reads — that opens directly in
Benchling or SnapGene.

In [ ]:
#@title Download the design files {display-mode: "form"}
import os
from IPython.display import HTML, display

if not result["paths"]:
    display(HTML('<div style="opacity:.7">Set <code>write_files</code> to True in the '
                 'parameters cell and re-run.</div>'))
else:
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None

    LABEL = {"oligo_csv": "Order sheet (CSV)", "oligo_fasta": "Oligos (FASTA)",
             "gene_fasta": "Assembled gene (FASTA)", "genbank": "Annotated GenBank"}
    rows = "".join(
        f'<tr><td style="padding:.35em .9em">{LABEL.get(k, k)}</td>'
        f'<td style="padding:.35em .9em;font-family:ui-monospace,monospace;font-size:12px;'
        f'opacity:.7">{os.path.basename(p)}</td>'
        f'<td style="padding:.35em .9em;text-align:right;font-variant-numeric:tabular-nums;'
        f'opacity:.7">{os.path.getsize(p):,} B</td></tr>'
        for k, p in result["paths"].items())
    display(HTML(f'<table style="font-family:ui-sans-serif,system-ui,sans-serif;'
                 f'font-size:13px;border-collapse:collapse">{rows}</table>'))

    if colab_files:
        for p in result["paths"].values():
            colab_files.download(p)

## Does this target also exist in the chloroplast?

A PPR cannot tell which copy of a sequence you meant. If your target also occurs in an
endogenous chloroplast transcript, the protein binds there too and stops being specific to
your construct.

**A PPR binds RNA, so only transcripts count.** A match in a non-transcribed region is not
an RNA off-target, and neither is a reverse-complement match in DNA — the transcript from
that locus carries the other sequence. The scan reports both tiers so you can tell them
apart.

The *Chlamydomonas* chloroplast is 203,828 bases and 34.5% GC, with 109 annotated
transcripts covering 43.5% of it. Over 200 random targets of each length:

| target length | in genomic DNA | **in a transcript** |
|---|---|---|
| 9 nt | 97 of 200 (48%) | **32 of 200 (16%)** |
| 14 nt | 0 | 0 |
| 19 nt | 0 | 0 |

A nine-base sequence is not rare enough in a 204 kb genome; a fourteen-base one is. If a
target comes back flagged in the transcript tier, lengthening it is the reliable fix.

Occurrence is a *necessary* condition for an off-target interaction, never a sufficient
one. This reports sequence, not affinity — no binding is predicted.

In [ ]:
#@title Off-target check — does the host already contain this sequence? {display-mode: "form"}
from clippr.offtarget import architecture_advice, load_genome, load_transcripts, report, scan

genome = load_genome()
transcripts = load_transcripts()
gc = 100 * (genome.count("G") + genome.count("C")) / len(genome)
print(f"host: Chlamydomonas reinhardtii chloroplast, {len(genome):,} bp, {gc:.1f}% GC")
print()

print("expected occurrences by chance, for an average target:")
for n, e in architecture_advice(genome).items():
    print(f"  {n:>2}-nt target : {e:8.3f}")

print()
print(f"{len(transcripts)} annotated transcripts, "
      f"{sum(len(x.sequence) for x in transcripts):,} nt "
      f"({100*sum(len(x.sequence) for x in transcripts)/len(genome):.1f}% of the genome)")
print()
print(report([scan(target_rna, genome, transcripts)]))

---

## Do you already own the parts?

Everything above designs DNA to be **synthesised**. But the GRASP authors deposited a
42-plasmid kit, and a lab that holds it can assemble many PPRs from parts it already has. For
such a lab, "order 906 nt of new DNA" is the wrong answer to a question with a cheaper one.

So a target gets two realisation routes, judged by the same audit:

| route | what it costs | what it constrains |
|---|---|---|
| **de novo synthesis** | new DNA | nothing — full synonymous freedom |
| **GRASP module kit** | nothing, if you hold the kit | fixed to the deposited parts |

The kit turns out to be sized exactly for its job. Modules chain by their Golden Gate
overhangs through a graph with a single branch point, one run of `B C D` plus a linker
contributes five modules, and an *n*-base target needs *n+1* modules — so 9, 14 and 19 bases
need two, three and four sub-assemblies. Each internal join consumes one linker pair, 19S needs
three, and the kit contains exactly three. It cannot build anything longer, and the cell below
says so plainly when asked.

> Module identity, overhangs and plate positions are derived from **Dennis et al. 2025
> Supplementary Table S1**, and the selection reproduces the module lists published in Table S2
> for 28 of 28 internally consistent variants. This selects the PPR modules only — not the
> acceptor plasmids or the rest of the transcriptional unit.

In [ ]:
#@title GRASP kit route — can you build this from parts you own? {display-mode: "form"}
from clippr import parts_report, select_parts

plan = select_parts(target_rna)
print(parts_report(plan))

---

## Is this design good, relative to the alternatives?

The design above ranks assembly plans by predicted ligation fidelity and keeps the first one
whose coding sequence satisfies every constraint. The rest are discarded unexamined — so it
cannot tell you whether the answer was a good one.

The cell below carries several plans **all the way through** codon optimisation and QC, then
picks between finished designs using a stated priority order rather than hidden weights:

1. every constraint satisfied and QC not FAIL
2. predicted fidelity within a tolerance of the best feasible value
3. prefer QC PASS, then the higher optimiser score
4. tie-break on fidelity

You get one answer plus the alternatives and what each would cost — not a trade-off plot to
arbitrate.

> **Measured caveat.** Across 9S, 14S and 19S, predicted fidelity came out *identical for every
> candidate* and QC passed for every candidate: fidelity is capped by the destination overhang
> pair, and the infeasible overhangs were already removed earlier. So in this configuration the
> ranking is effectively decided by sequence quality alone. It still improves on the
> first-feasible plan for all three architectures — but this is not a multi-objective optimiser,
> and calling it one would be wrong.

In [ ]:
#@title Explore the alternatives, then justify one {display-mode: "form"}
search_budget = 6  #@param {type:"slider", min:2, max:12, step:1}

from clippr import design_searched

searched = design_searched(target_rna, organism=organism, codon_table=None,
                           enzyme_profile=enzyme_profile, seed=seed,
                           check_offtarget=False, budget=search_budget)
print(f"exploring changed the chosen design: {searched['differs_from_oneshot']}")
print()
print(searched["certificate"])

---

## Designing a whole library

For a set of regulators, what matters is **orthogonality**: PPRᵢ must bind UTRᵢ and not
UTRⱼ. The matrix below is the pairwise distance between targets — larger is better
separated. Targets that sit close together risk one PPR binding another's UTR.

> `orthogonal.py` is a **capability, not a validated result**. Every other part of this
> package is checked against a 200-design corpus; this one has unit tests only, because no
> ground truth for it exists.

In [ ]:
#@title Design the whole library {display-mode: "form"}
targets = "AAAAUGUGG, GCUAAAGAC, UUACACGUG"  #@param {type:"string"}

from clippr import design_library

target_list = [t.strip().upper() for t in targets.split(",") if t.strip()]
lib = design_library(target_list, codon_table=None, organism=organism,
                     enzyme_profile=enzyme_profile, seed=seed,
                     check_offtarget=check_offtarget,
                     outdir="clippr_library" if write_files else None,
                     on_progress=lambda i, n, t: print(f"  {i}/{n}  {t}", flush=True))

print()
print(lib.summary())
print()
print(lib.crosstalk())

In [ ]:
#@title Library QC table {display-mode: "form"}
lib.qc_table()

### DNA shared between members

Cross-talk asks whether two PPRs could bind each other's **target**. This asks whether two
**genes** share enough identical DNA to recombine — a different question with a different
answer. Every member of a PPR library carries the same scaffold, so it is never trivially no.

Measured on five 9S designs, every pair shared at least **59 nt**, and every one of those
stretches began at position 0 in both members: the fixed 23-residue N-terminal scaffold, which is
protein-identical by construction and gets the same codons every time. Most, but not all — two
pairs of a six-member library share 62 nt inside the repeat body, at positions 663/663 and
597/318, with no scaffold involved.

`diversify_library` gives each member its own synonymous encoding of that scaffold, locked in
place. On those five designs: longest shared stretch **107 → 47 nt**, pairs over the 50 nt
threshold **10 → 0**. It is deterministic in the member index, so the library stays reproducible,
and a diversified member is accepted only when it is no worse than the one it replaces. **At six
members it reaches only 77 nt and leaves 2 of 15 pairs above the threshold** — part of the
residue lives in the repeat body rather than the scaffold, and this retry schedule did not
reach it.

A shared stretch is a *necessary* substrate for recombination, never a prediction that it will
happen, and 50 nt is a rule of thumb rather than a measured constant for this host.

In [ ]:
#@title Homology — DNA shared between library members {display-mode: "form"}
print(lib.homology())

### Cross-talk, in two tiers

Sequence separation is one question; *predicted binding* is a different one, and mixing
them would smuggle an unvalidated model into a hard criterion. So they stay apart:

| tier | what it is | status |
|---|---|---|
| **A — Hamming distance** | two targets differ in *k* of *n* positions | **the hard criterion, and the only thing that gates** |
| **B — predicted affinity** | the PPR designed for A, scored against B | an annotation; gates nothing |

Tier A makes no biological claim — it says two sequences differ in *k* places, which is
geometry, and stays true whatever anyone later learns about PPR binding.

Tier B needs a PPR specificity table. **CLIPPR does not ship one**: the available table
(Yan *et al.*, distributed with [PPRmatcher](https://github.com/ian-small/PPRmatcher))
carries no licence, so redistributing it would be a rights problem however useful it is.
It also comes from **P-type** PPR experiments, while this scaffold is **S-type** — all four
codes GRASP uses do score their cognate base highest in it, which is reassuring, but a
P-type model has not been shown to apply here. A high score means *look*, never *fail*.

Leave the path blank and you get tier A alone, which is the criterion that gates anyway.

In [ ]:
#@title Two-tier cross-talk — separation gates, affinity annotates {display-mode: "form"}
ppr_score_table = ""  #@param {type:"string"}

from clippr import compare_tiers, load_ppr_scores

scores = load_ppr_scores(ppr_score_table) if ppr_score_table.strip() else None
print(lib.crosstalk(scores=scores))

if scores:
    cmp = compare_tiers(lib.targets, scores)
    print()
    print(f"pairs flagged by separation:      {len(cmp['hamming_flagged'])}")
    print(f"pairs flagged by predicted affinity: {len(cmp['affinity_flagged'])}")
    print(f"the model points somewhere separation does not: {cmp['tiers_disagree']}")
    print()
    print("A disagreement is a reading recommendation, not a failed design —")
    print("tier A alone decides what this library accepts.")

---

# The reusable-inventory route

Everything above designs a **new coding sequence per target**. This section is the other
route: take the deposited GRASP module kit, recode it once for your host, then *compile*
targets from it — the DNA is ordered once and reused.

The two routes answer different questions. Synthesis gives you any target the PPR code can
express. The kit gives you the targets its modules can spell, far more cheaply, because you
are assembling parts you already have.

**This route needs Supplementary Table S1.** It holds the module insert sequences and is not
distributed with this package — see `NOTICE.md`. Upload it below. Every step after the recode
works from a *saved inventory*, which carries its own sequences, so you need Table S1 once.

In [ ]:
#@title Inventory 1 - load the deposited kit {display-mode: "form"}
#@markdown Upload **Supplementary Table S1** (`.xlsx`), then put its filename here.
table_s1 = "Table S1.xlsx"  #@param {type:"string"}

import json as _json
from pathlib import Path

from clippr import inventories as inv
from clippr import workflow as w

# This route resolves its own codon table, from the same organism and file chosen in the
# design form above. It cannot borrow the name `table`: the fragment-table cell rebinds that
# to a display DataFrame, so every call here was handed a DataFrame and raised. The name is
# distinct so a later display cell cannot shadow it again.
from clippr import constants as _C
from clippr.design import _resolve_codon_table as _resolve

_host_code = genetic_code_override or _C.ORGANISMS[organism][1]
host_codon_table = _resolve(codon_table_file or None, organism, _host_code)
print(f"codon table for {organism}, genetic code {_host_code}: "
      f"{len(host_codon_table)} amino acids")

deposited = None
if not Path(table_s1).is_file():
    print(f"{table_s1} not found. Upload it with the folder icon in the sidebar,")
    print("or use the upload cell near the top of this notebook.")
else:
    deposited = inv.load_deposited(table_s1)
    print(f"loaded {len(deposited)} modules  |  version {deposited.version}")
    print(f"structural problems: {inv.validate(deposited) or 'none'}")

    notes = inv.synthesis_profile_notes(deposited)
    if notes:
        print()
        print("Modules outside CLIPPR's synthesis profile, before you order anything:")
        for module_id, why in notes.items():
            print(f"   {module_id}: {'; '.join(why)}")
        print()
        print("That limit is CLIPPR's engineering choice, not a published rule, so this")
        print("is a statement about fit -- not a defect in the deposited kit.")

## Recode the kit for your host

Every module is rewritten to use your host's preferred codons while **the protein, the length
and both four-base interfaces stay exactly as they were**. A recoded module therefore drops
straight into an assembly built from unrecoded neighbours.

A module that cannot be improved keeps its original sequence and is reported as *unchanged*.
That is a delivered module, not a gap.

In [ ]:
#@title Inventory 2 - recode for your host, interfaces frozen {display-mode: "form"}
label = "my-host"  #@param {type:"string"}
candidate_seeds = 4  #@param {type:"slider", min:1, max:8, step:1}
recode_seconds = 600  #@param {type:"integer"}

recoded = None
if deposited is not None:
    recoded = w.recode_for_host(table_s1, host_codon_table, "out/inventory", label=label,
                                seeds=candidate_seeds, wall_seconds=recode_seconds)
    print(recoded.summary)
    print(f"saved to {recoded.artefacts['inventory']}")

    report = _json.loads(Path(recoded.artefacts["report"]).read_text(encoding="utf-8"))
    scored = [o for o in report["objectives"].values() if "cai_after" in o]
    if scored:
        before = sum(o["cai_before"] for o in scored) / len(scored)
        after = sum(o["cai_after"] for o in scored) / len(scored)
        print()
        print(f"mean CAI {before:.4f} -> {after:.4f} over {len(scored)} modules")
        print()
        print("CAI predicts how well codons match the host. It is not a measurement of")
        print("expression, and a large CAI gain is not a proportional expression gain.")

## Compile targets from the inventory

Give it targets; it returns the module list, the assembled product and every junction. A
target the kit cannot spell is reported with its reason rather than raising.

The product is **joined module inserts only** — no acceptor backbone, so it is not an
expression construct.

In [ ]:
#@title Inventory 3 - compile targets {display-mode: "form"}
inventory_targets = "AAAAUGUGG, UUACACGUGCGUAC"  #@param {type:"string"}
fusion_site = "AATG"  #@param ["AATG", "AGGT"]

wanted = [t.strip().upper().replace("T", "U")
          for t in inventory_targets.split(",") if t.strip()]
compiled = None
if recoded is not None:
    compiled = w.load_and_compile(recoded.artefacts["inventory"], wanted,
                                  "out/compiled", fusion_site=fusion_site)
    print(compiled.summary)
    for failure in compiled.failures:
        print(f"   could not build {failure['target']}: {failure['reason']}")

    products = _json.loads(
        Path(compiled.artefacts["compiled_targets"]).read_text(encoding="utf-8"))
    for name, product in products.items():
        print(f"   {name}: {product['product_nt']} nt, "
              f"{len(product['modules'])} modules, {product['reactions']} reaction(s)")

## Optimise the inventory as a collection

Recoding treats each module alone, so it cannot see what the modules share *with each other*.
This step can trade a little codon adaptation in one module for less shared sequence across
the set.

`greedy` is the default: it accepts only improvements, and over 18 matched runs it reaches
within 2% of what `anneal` achieves on shared sequence while making a third as many changes.
`anneal` explores further and gives up somewhat more codon adaptation for a small further
gain. `random` is a control — a method that cannot beat it has not been shown to work.

All three are **reproducible for a fixed seed and settings**, not seed-free deterministic:
the candidate replacements come from a seeded generator in every mode.

In [ ]:
#@title Inventory 4 - optimise the collection {display-mode: "form"}
mode = "greedy"  #@param ["greedy", "anneal", "random"]
max_proposals = 168  #@param {type:"integer"}

optimised = None
if recoded is not None:
    optimised = w.optimise_collection(recoded.artefacts["inventory"], host_codon_table,
                                      "out/collection", mode=mode,
                                      max_proposals=max_proposals, wall_seconds=600)
    print(optimised.summary)
    search = _json.loads(Path(optimised.artefacts["report"]).read_text(encoding="utf-8"))
    print(f"   adaptation {search['incumbent']['adaptation']} -> "
          f"{search['final']['adaptation']}")
    print(f"   shared k-mers per pair {search['incumbent']['sharing_per_pair']} -> "
          f"{search['final']['sharing_per_pair']}")
    print()
    print("The sharing term steers the search. It is not a recombination probability,")
    print("and lowering it does not necessarily shorten the worst shared tract.")

## Explore junction trade-offs

The kit's internal junction overhangs can be swapped for other four-base sequences encoding
the same protein. Different choices give different predicted assembly fidelity, so there is a
real trade-off to look at rather than a single answer.

⚠️ **Choosing an alternative changes the interface version.** Every target must be recompiled
from the selected inventory, and modules from the previous version cannot be mixed with it.

The result is the **observed** front — nondominated among the candidates actually evaluated.
A bounded search does not enumerate the space, so this is not the globally optimal front.

In [ ]:
#@title Inventory 5 - explore junction trade-offs {display-mode: "form"}
max_evaluations = 100  #@param {type:"integer"}

front = None
if recoded is not None:
    front = w.explore_interfaces(recoded.artefacts["inventory"], wanted, host_codon_table,
                                 "out/interfaces", max_evaluations=max_evaluations,
                                 wall_seconds=600)
    print(front.summary)
    print()
    print(f"incumbent  : {front.data['incumbent']}")
    print(f"recommended: {front.data['recommended']}")
    print()
    print(f"policy: {front.data['recommendation_policy']}")
    if front.data["fidelity_degenerate"]:
        print()
        print("Every candidate scored the same fidelity -- no trade-off to make here.")
    print()
    print(front.data["selecting_an_alternative"])

## Commit to a choice

Exploring a front does nothing until you pick one. This saves the chosen candidate as a real
inventory and builds the order items from *it*, so what you order is what you chose.

The front is **bound** to the inventory it was measured against. Selecting it against a
different one is refused rather than silently reporting the wrong inventory's numbers.

`incumbent` is a perfectly good choice when the front offers nothing worth an interface change.

In [ ]:
#@title Inventory 6 - commit to a choice and rebuild {display-mode: "form"}
choice = "recommended"  #@param ["recommended", "incumbent"]

selected = recompiled = order_items = None
if front is not None:
    selected = w.select_interface(recoded.artefacts["inventory"],
                                  front.artefacts["front"], "out/selected",
                                  choice=choice)
    print(selected.summary)

    recompiled = w.load_and_compile(selected.artefacts["inventory"], wanted,
                                    "out/recompiled")
    print(recompiled.summary)

    order_items = w.order_items_for(selected.artefacts["inventory"], "out/order_items")
    print(order_items.summary)

## Plan the order

Eligibility, pooling and export against a written product profile.

The order items are **assembly-ready substrates**, not bare inserts: each is wrapped so BbsI
releases the level-0 fragment with the correct exposed ends, and each is digest-verified
before export. An A-module insert starts with its own `AATG` fusion site, while the substrate
must expose `CTCA` — ordering the insert would order something that cannot assemble.

**A local check is not vendor approval.** The shipped profile carries this project's
historical price constants and deliberately **refuses to price**, because an undated number
presented as current is worse than no number at all. Eligibility, pooling and export work
without one; supply a dated profile to get an estimate.

In [ ]:
#@title Inventory 7 - eligibility and pool plan {display-mode: "form"}
#@markdown Leave blank for the current published oPools rules (unpriced).
#@markdown Supply a dated profile JSON to get a cost estimate.
profile_file = ""  #@param {type:"string"}

from clippr.ordering import CURRENT_OPOOL_50PMOL, load_profile

profile = load_profile(profile_file) if profile_file.strip() else CURRENT_OPOOL_50PMOL

# The order comes from the inventory you selected, never from whatever happens to be left in
# notebook memory. Taking sequences from the synthesis route's `lib` here meant a user could
# explore one inventory and order something unrelated to it.
order = None
if order_items is None:
    print("Run the cells above first -- the order is built from the selected inventory.")
else:
    items = _json.loads(Path(order_items.artefacts["order_items"]).read_text(
        encoding="utf-8"))
    print(f"ordering {len(items)} assembly-ready substrates from "
          f"{selected.data['to_version']}")
    print(f"exposed ends: {order_items.data['exposed_ends']}")
    order = w.plan_order(items, profile, "out/order")
    print(order.summary)
    for failure in order.failures:
        print(f"   {failure['stage']}: {failure['reason']}")

    cost = order.data["cost"]
    if cost["available"]:
        print(f"   estimate {cost['total']} {cost['currency']} "
              f"(priced {cost['priced_on']}) -- an estimate, not a quote")
    else:
        print(f"   cost unavailable: {cost['reason']}")
    print()
    for rule in order.data["eligibility"]["unresolved"]:
        print(f"   unresolved: {rule}")

## Take the package

One directory holding the manifest, every task's result, the artefacts and — importantly —
every failure. Artefact paths are recorded relative to the package, so it can be moved or
shared without carrying a machine-specific path along with it.

In [ ]:
#@title Inventory 8 - download the result package {display-mode: "form"}
done = [r for r in (recoded, compiled, optimised, front, selected, recompiled,
                    order_items, order) if r is not None]
if not done:
    print("Nothing to package yet -- run the cells above.")
else:
    package = w.write_package(done, "out/package",
                              inputs={"table_s1": table_s1, "targets": wanted})
    print(f"wrote {package}")
    summary = _json.loads(Path(package).read_text(encoding="utf-8"))
    print(f"   {len(summary['tasks'])} tasks, ok={summary['ok']}, "
          f"{len(summary['failures'])} failure(s)")
    print()
    print(summary["scope"])

    try:
        import shutil

        from google.colab import files
        shutil.make_archive("clippr_package", "zip", "out")
        files.download("clippr_package.zip")
    except ImportError:
        print()
        print("Not running in Colab -- the package is in out/.")